# 04. Game of Thrones — Communities and a Publication-Ready Figure

Notebook 03 ended on a disciplined result: the *Game of Thrones* Book 1 network drawn on an **overlap-free Graphviz layout** (`neato` + `-Goverlap=prism`), with every label decision verified by a **measurable clutter metric**.

This notebook picks up from that layout and asks the next question: **once we know the network has community structure, how do we *see* it?** We'll discover that the overlap-free shared layout — lovely as it is for labels — actively hides the community signal, and we'll build an alternative layout that brings it front-and-centre.

**Roadmap**

1. **Preamble** — rebuild the shared layout and detect communities.
2. **Section A** — draw the shared layout coloured by community and observe why it fails the communities story.
3. **Section B** — build a community-aware ring layout that makes the group structure unmistakable.
4. **Section C** — return to labels, this time on the community substrate.
5. **Section D** — assemble everything into one publication-ready composite.

**Learning goals**

- see why a single "best" layout cannot serve every analytical question
- detect and name communities with weighted Louvain
- encode community membership via position (ring layout + concentric-ring packing), not just colour
- combine community, centrality, and labelling without losing any one of them


In [ ]:
from pathlib import Path
import sys
from textwrap import fill

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

cwd = Path.cwd().resolve()
tutorials_dir = cwd.parent if cwd.name == "04-networks" else cwd / "tutorials"
if str(tutorials_dir) not in sys.path:
    sys.path.insert(0, str(tutorials_dir))

from dataviz_utils import *

set_seeds()
TEXT = apply_teaching_rc(grid=False, font_base=11.5)
FIG = make_figure_size_scale(
    focus=(7.2, 7.2),
    standard=(8.2, 5.2),
    wide=(11.8, 6.6),
)

NOTEBOOK_DIR = tutorials_dir / "04-networks"
DATA_DIR = NOTEBOOK_DIR / "data"


## Helpers

Two families of helpers support the rest of the notebook:

**Shared drawing primitives** (next cell) — style constants used by Section A (`LINK_WIDTH`, `LINK_ALPHA`, `SOCIAL_EDGE_COLOR`) plus two small utilities: `fit_network_axes` and `add_panel_header`.

**Community-aware drawing** (the cell after) — layout + reusable drawing blocks that Sections B, C, and D share:

- `detect_communities`, `pick_community_anchors`, `name_communities`, `build_community_palette` — partition and label.
- `community_aware_layout` — the two-stage Gephi-style ring layout.
- `draw_community_discs` — soft circular backgrounds.
- `draw_community_ring` — the full substrate (discs + intra-community edges + inter-community edges + nodes), used by **B, C, and D** with identical parameters so the three figures are visually consistent.
- `add_outer_ring_labels` — anchor labels placed on a single outer ring beyond every disc, shared by **C and D**.
- `add_community_legend` — right-column size-sorted legend, shared by **B and C**.

Factoring these three drawing blocks out means each section's code reads as a short composition: *substrate → optional labels → optional legend*, with no duplicated style values to drift out of sync.


In [ ]:
# Edge style constants — used by Section A's shared-layout plot.
LINK_WIDTH = 0.7
LINK_ALPHA = 0.6
SOCIAL_EDGE_COLOR = "#C8CED8"


def fit_network_axes(ax, pos, pad=0.08):
    """Fit axes limits around a network's node positions with a proportional pad."""
    coords = np.asarray(list(pos.values()), dtype=float)
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    span = np.maximum(maxs - mins, 1e-6)
    ax.set_xlim(mins[0] - span[0] * pad, maxs[0] + span[0] * pad)
    ax.set_ylim(mins[1] - span[1] * pad, maxs[1] + span[1] * pad)
    return ax


def add_panel_header(ax, title=None, subtitle=None):
    """Attach a left-aligned title + subtitle above `ax`, shrinking the
    plotting area by exactly the header's measured height."""
    fig = ax.figure
    pos = ax.get_position()
    fig_width, fig_height = fig.get_size_inches()

    def pts_to_fig_y(points):
        return (points / 72.0) / fig_height

    def width_to_chars(width_fraction, chars_per_inch=11):
        panel_width_in = max(fig_width * width_fraction, 2.0)
        return max(28, int(panel_width_in * chars_per_inch))

    wrapped_title = fill(title, width=width_to_chars(pos.width, chars_per_inch=9)) if title else None
    wrapped_subtitle = fill(subtitle, width=width_to_chars(pos.width)) if subtitle else None
    title_lines = wrapped_title.count("\n") + 1 if wrapped_title else 0
    subtitle_lines = wrapped_subtitle.count("\n") + 1 if wrapped_subtitle else 0

    title_line_pts = TEXT["panel_title"] * 1.08
    subtitle_line_pts = TEXT["annotation"] * 1.22
    top_margin_pts = 2
    gap_pts = 3 if wrapped_title and wrapped_subtitle else 0
    bottom_margin_pts = 4 if (wrapped_title or wrapped_subtitle) else 0

    header_pts = top_margin_pts + bottom_margin_pts
    if wrapped_title:
        header_pts += title_lines * title_line_pts
    if wrapped_subtitle:
        header_pts += gap_pts + subtitle_lines * subtitle_line_pts

    min_header_pts = 18 if (wrapped_title or wrapped_subtitle) else 0
    header_height = min(pts_to_fig_y(max(header_pts, min_header_pts)), pos.height * 0.18)
    new_height = max(pos.height - header_height, pos.height * 0.68)
    ax.set_position([pos.x0, pos.y0, pos.width, new_height])

    header_top_y = pos.y0 + pos.height - pts_to_fig_y(top_margin_pts)
    current_top = header_top_y

    if wrapped_title:
        fig.text(pos.x0, current_top, wrapped_title, ha="left", va="top",
                 fontsize=TEXT["panel_title"], color="#17212B")
        current_top -= pts_to_fig_y(title_lines * title_line_pts + gap_pts)

    if wrapped_subtitle:
        fig.text(pos.x0, current_top, wrapped_subtitle, ha="left", va="top",
                 fontsize=TEXT["annotation"], color=DV_PALETTE["gray"])

    ax.set_axis_off()
    return ax


In [ ]:
# =============================================================================
# Community detection and naming
# =============================================================================

def detect_communities(G, *, seed=42, weight="weight"):
    """Weighted Louvain, sorted by size (descending). Returns the sorted
    list of communities and a {node: community_id} lookup."""
    comms = nx.community.louvain_communities(G, seed=seed, weight=weight)
    comms = sorted(comms, key=len, reverse=True)
    node_to_cid = {node: cid for cid, community in enumerate(comms) for node in community}
    return comms, node_to_cid


def build_community_palette(n_total):
    return {cid: CATEGORICAL_PALETTE[cid % len(CATEGORICAL_PALETTE)]
            for cid in range(n_total)}


def pick_community_anchors(G, communities):
    wd = dict(G.degree(weight="weight"))
    return [max(community, key=lambda node: (wd[node], G.degree(node), node))
            for community in communities]


COMMUNITY_NAMES_BY_ANCHOR = {
    "Eddard-Stark":       "King's Landing Court",
    "Tyrion-Lannister":   "Riverlands & the Eyrie",
    "Jon-Snow":           "The Night's Watch",
    "Sansa-Stark":        "Sansa in King's Landing",
    "Daenerys-Targaryen": "Daenerys & the Dothraki",
    "Bran-Stark":         "Winterfell",
    "Waymar-Royce":       "Prologue — Beyond the Wall",
    "Danwell-Frey":       "House Frey",
}


def name_communities(anchors, mapping=COMMUNITY_NAMES_BY_ANCHOR):
    return [mapping.get(anchor, f"Arc around {anchor}") for anchor in anchors]


# =============================================================================
# Two-stage community-aware layout
# =============================================================================

def _concentric_rings_disk(n, *, inward_angle=0.0, rotation=0.0):
    """Concentric rings inside the unit disk, ordered so the caller can put
    high-degree members on the rim facing the figure centre.

    Priority order returned:
    1. Outer ring first, angles sorted by closeness to ``inward_angle``.
    2. Inner rings next (same angular sort).
    3. Disk centre last.
    """
    if n <= 0:
        return np.zeros((0, 2))
    if n == 1:
        return np.array([[0.0, 0.0]])
    if n <= 3:
        # First vertex on the inward-facing side so priority slot 0 (the
        # highest-degree member) lands closest to the figure centre.
        thetas = inward_angle + 2 * np.pi * np.arange(n) / n
        return np.column_stack([np.cos(thetas), np.sin(thetas)])

    # Ring capacities: 1 centre + 6 * k per ring k.
    ring_sizes = [1]
    k = 1
    while sum(ring_sizes) < n:
        ring_sizes.append(6 * k)
        k += 1
    if sum(ring_sizes) > n:
        ring_sizes[-1] -= (sum(ring_sizes) - n)
        if ring_sizes[-1] == 0:
            ring_sizes.pop()

    n_rings = max(len(ring_sizes) - 1, 1)

    def _sort_ring_angles(size, ring_idx):
        base_offset = rotation + 0.5 * ring_idx
        base_angles = [2 * np.pi * i / size + base_offset for i in range(size)]
        def ang_dist(a):
            delta = (a - inward_angle + np.pi) % (2 * np.pi) - np.pi
            return abs(delta)
        return sorted(base_angles, key=ang_dist)

    positions = []
    for ring_idx in range(len(ring_sizes) - 1, 0, -1):
        size = ring_sizes[ring_idx]
        if size <= 0:
            continue
        radius = ring_idx / n_rings
        for a in _sort_ring_angles(size, ring_idx):
            positions.append((radius * np.cos(a), radius * np.sin(a)))
    positions.append((0.0, 0.0))
    return np.asarray(positions[:n])


def _order_communities_on_ring(meta, communities):
    """Greedy TSP-style ordering: start with the largest community, then
    keep appending the most-connected not-yet-placed neighbour."""
    order = [max(meta.nodes(), key=lambda c: len(communities[c]))]
    remaining = set(meta.nodes()) - {order[0]}
    while remaining:
        current = order[-1]
        best = max(remaining,
                   key=lambda c: meta[current][c]["weight"] if meta.has_edge(current, c) else 0)
        order.append(best)
        remaining.remove(best)
    return order


def _solve_variable_ring(order, radii_target, gap_fraction):
    """Solve for ring radius R and per-community angular position so that
    adjacent disks just fit without overlap."""
    n = len(order)
    gap = 1.0 + gap_fraction
    adj_sums = [(radii_target[order[i]] + radii_target[order[(i + 1) % n]]) * gap
                for i in range(n)]
    R_low = max(adj_sums) / 2.0 * 1.0001
    R_high = max(sum(adj_sums), R_low * 2)

    def sum_arcsin(R):
        return sum(np.arcsin(min(s / (2 * R), 1.0)) for s in adj_sums)

    for _ in range(80):
        R_mid = 0.5 * (R_low + R_high)
        if sum_arcsin(R_mid) > np.pi:
            R_low = R_mid
        else:
            R_high = R_mid
    R = 0.5 * (R_low + R_high)

    thetas = {}
    cumulative = -np.pi / 2
    for i, cid in enumerate(order):
        thetas[cid] = cumulative
        cumulative += 2 * np.arcsin(min(adj_sums[i] / (2 * R), 1.0))
    return R, thetas


def community_aware_layout(G, node_to_cid, communities, *,
                           size_proportional=True, gap_fraction=0.14):
    """Two-stage community-aware layout.

    Stage 1 — communities on a ring with variable angular spacing so that
    larger communities get more room without overlapping their neighbours.
    Stage 2 — concentric-ring packing within each community, with the
    highest-degree member on the rim facing the figure centre.
    """
    cids = sorted(set(node_to_cid.values()))

    # Meta-graph of inter-community weight — drives the TSP-style ordering.
    meta = nx.Graph()
    for cid in cids:
        meta.add_node(cid)
    for u, v, d in G.edges(data=True):
        cu, cv = node_to_cid[u], node_to_cid[v]
        if cu != cv:
            w = float(d.get("weight", 1))
            if meta.has_edge(cu, cv):
                meta[cu][cv]["weight"] += w
            else:
                meta.add_edge(cu, cv, weight=w)

    order = _order_communities_on_ring(meta, communities)
    sizes = {cid: len(communities[cid]) for cid in cids}
    target = ({cid: float(np.sqrt(sizes[cid])) for cid in cids}
              if size_proportional else {cid: 1.0 for cid in cids})

    ring_radius, thetas = _solve_variable_ring(order, target, gap_fraction)
    super_pos = {cid: (ring_radius * np.cos(thetas[cid]),
                       ring_radius * np.sin(thetas[cid])) for cid in cids}
    radii = dict(target)

    wd = dict(G.degree(weight="weight"))
    pos = {}
    for cid in cids:
        members = sorted([n for n in G.nodes() if node_to_cid[n] == cid],
                         key=lambda n: (-wd[n], n))  # descending degree
        cx, cy = super_pos[cid]
        inward_angle = float(np.arctan2(-cy, -cx))
        disk = _concentric_rings_disk(len(members), inward_angle=inward_angle)
        r = radii[cid]
        for node, (dx, dy) in zip(members, disk):
            pos[node] = (float(cx + dx * r), float(cy + dy * r))
    return pos, super_pos, radii


# =============================================================================
# Shared drawing helpers (used by Sections B, C, D)
# =============================================================================

# Radius multiplier for the soft discs. Lives here because both
# `draw_community_discs` and `add_outer_ring_labels` need to agree on it
# to keep labels outside the visible disc border.
DISC_RADIUS_SCALE = 1.12


def draw_community_discs(ax, super_pos, radii, palette, *, alpha=0.20,
                         edge_alpha=0.55, zorder=0):
    """Soft filled + outlined background disc per community."""
    for cid, (cx, cy) in super_pos.items():
        r = radii[cid] * DISC_RADIUS_SCALE
        colour = palette[cid]
        ax.add_patch(mpatches.Circle(
            (cx, cy), radius=r,
            facecolor=lighten_color(colour, 0.85),
            edgecolor="none", alpha=alpha, zorder=zorder,
        ))
        ax.add_patch(mpatches.Circle(
            (cx, cy), radius=r,
            facecolor="none",
            edgecolor=lighten_color(colour, 0.35),
            linewidth=0.9, alpha=edge_alpha, zorder=zorder + 1,
        ))
    return ax


def draw_community_ring(ax, G, *, pos, super_pos, radii, palette,
                        node_to_cid, intra_by_cid, inter_edges, inter_widths,
                        node_sizes):
    """Full community-ring substrate: soft discs, community-tinted intra
    edges, slate inter-community edges, and community-coloured nodes.

    Every section that uses the ring layout (B, C, D) calls this with the
    same arguments, so the three figures stay visually in sync.
    """
    draw_community_discs(ax, super_pos, radii, palette,
                         alpha=0.20, edge_alpha=0.40)

    for cid, (eds, wds) in intra_by_cid.items():
        if not eds:
            continue
        nx.draw_networkx_edges(
            G, pos, ax=ax,
            edgelist=eds, width=wds,
            edge_color=[lighten_color(palette[cid], 0.35)] * len(eds),
            alpha=0.85,
        )

    nx.draw_networkx_edges(
        G, pos, ax=ax,
        edgelist=inter_edges, width=inter_widths,
        edge_color="#4A5560", alpha=0.30,
    )

    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=[palette[node_to_cid[n]] for n in G.nodes()],
        node_size=[node_sizes[n] for n in G.nodes()],
        edgecolors="black", linewidths=0.7,
    )
    return ax


def add_outer_ring_labels(ax, *, super_pos, radii, pos, anchors, palette,
                          pad=1.0):
    """Anchor labels on a single outer ring beyond every community disc.

    The ring radius is the farthest any disc reaches from the figure
    centre (plus ``pad``), so small and large discs alike keep their
    label outside the envelope — impossible to achieve by anchoring each
    label to its own node, because anchor nodes sit at different
    distances from the community centre.
    """
    outer_envelope = max(
        np.hypot(cx, cy) + radii[cid] * DISC_RADIUS_SCALE
        for cid, (cx, cy) in super_pos.items()
    )
    label_ring_radius = outer_envelope + pad

    for cid, (cx, cy) in super_pos.items():
        anchor = anchors[cid]
        ax_node, ay_node = pos[anchor]
        norm = max(np.hypot(cx, cy), 1e-9)
        lx = cx / norm * label_ring_radius
        ly = cy / norm * label_ring_radius
        ax.annotate(
            anchor,
            xy=(ax_node, ay_node), xytext=(lx, ly),
            ha="center", va="center",
            fontsize=TEXT["direct_label"], fontweight="semibold",
            color="#17212B",
            bbox=dict(boxstyle="round,pad=0.36", fc="white",
                      ec=lighten_color(palette[cid], 0.40),
                      lw=1.4, alpha=0.98),
            arrowprops=dict(arrowstyle="-", color="#4A5560",
                            lw=1.0, alpha=0.85, shrinkA=6, shrinkB=6),
            zorder=25,
        )
    return ax


def add_community_legend(ax, *, communities, community_names, palette,
                         cids, loc="upper left", bbox_to_anchor=(1.005, 1.0)):
    """Right-column size-sorted community legend."""
    legend_order = sorted(cids, key=lambda c: -len(communities[c]))
    handles = [
        mpatches.Patch(
            facecolor=palette[cid], edgecolor="black", linewidth=0.6,
            label=f"{community_names[cid]}  ({len(communities[cid])})",
        )
        for cid in legend_order
    ]
    ax.legend(
        handles=handles, loc=loc, bbox_to_anchor=bbox_to_anchor,
        frameon=True,
        fontsize=TEXT["annotation"],
        title="Community (size)", title_fontsize=TEXT["annotation"],
        borderaxespad=0.0, handlelength=1.4, labelspacing=0.6,
    )
    return ax


def minmax_scale(values, low, high):
    """MinMax rescale a sequence of numbers to [low, high]."""
    arr = np.asarray(list(values), dtype=float).reshape(-1, 1)
    if np.allclose(arr, arr[0]):
        return np.full(arr.shape[0], (low + high) / 2)
    return MinMaxScaler(feature_range=(low, high)).fit_transform(arr).flatten()


## The Dataset

The edge list is identical to notebook 03: one row per pair of *Game of Thrones* Book 1 characters who interact, with a `weight` column counting those interactions.


In [ ]:
got_edges = pd.read_csv(DATA_DIR / "got" / "book1.csv")
got_edges.head(2)


In [ ]:
GOT = nx.from_pandas_edgelist(
    got_edges,
    source="Source",
    target="Target",
    edge_attr="weight",
    create_using=nx.Graph(),
)

print(f"{GOT.number_of_nodes()} characters, {GOT.number_of_edges()} weighted interactions")


## Reuse the Overlap-Free Layout From Notebook 03

Section A needs the overlap-free shared layout to demonstrate the "colour alone is not enough" problem. We do **not** recompute it — the same Graphviz call used in notebook 03 gives us an overlap-free geometry in which no two node markers overlap.

Sections B, C, and D will build a **different, community-aware** layout on top of the same graph, so `got_layout` is used only once, in Section A.


In [ ]:
got_layout = nx.nx_agraph.graphviz_layout(GOT, prog="neato", args="-Goverlap=prism")


## Preliminaries — Detect the Communities

Before we can compare layouts, we need the communities themselves. We run **weighted Louvain** once (the comparison at the end of notebook 03 showed it's tied for the highest modularity on this graph), sort the detected groups by size, and attach a narrative name to each.

Naming by anchor character (highest-weighted-degree member) makes the labels survive minor Louvain reshuffles across seeds or library versions: *"Sansa in King's Landing"* lands on Sansa's group regardless of tiny re-bucketings at the edges.


In [ ]:
communities, node_to_cid = detect_communities(GOT, seed=42)  # weighted Louvain

palette = build_community_palette(n_total=len(communities))
community_anchors = pick_community_anchors(GOT, communities)
community_names = name_communities(community_anchors)

community_overview = pd.DataFrame({
    "name": community_names,
    "anchor": community_anchors,
    "size": [len(c) for c in communities],
    "colour": [palette[i] for i in range(len(communities))],
})
community_overview


## Section A — The Shared Layout Hides the Communities

We inherit the overlap-free `got_layout` from notebook 03. It was built to solve a *labelling* problem: no two node markers overlap, so callouts land cleanly. But Graphviz doesn't know anything about communities — its objective is node-marker non-overlap, not clustered geometry.

To see the consequence, we're going to colour the *same* `got_layout` by community. If the layout respected community structure, we'd see distinct coloured regions. If it doesn't, we'll see a rainbow where groups are interleaved and the community signal is essentially invisible.


### Step 1 — Prepare the per-node community colour list

For each node we look up its community id (already set as a node attribute during detection) and index into the palette. This is the only ingredient we need to add on top of the baseline figure.


In [ ]:
community_fill = [palette[node_to_cid[n]] for n in GOT.nodes()]
community_fill[:6]  # spot-check: one colour per node


### Step 2 — Draw the shared layout, now coloured by community

Identical node markers to notebook 03, identical `got_layout` geometry — only the fill colour changes.


In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])

nx.draw_networkx_edges(
    GOT, got_layout, ax=ax,
    width=LINK_WIDTH, edge_color=SOCIAL_EDGE_COLOR, alpha=LINK_ALPHA,
)
nx.draw_networkx_nodes(
    GOT, got_layout, ax=ax,
    node_color=community_fill,
    node_size=36, edgecolors="black", linewidths=0.6,
)

fit_network_axes(ax, got_layout, pad=0.08)
add_panel_header(
    ax,
    "A. Shared layout, coloured by community",
    "The Graphviz overlap-free layout does not account for communities. Colours are scattered all over the figure — the group structure is invisible from position alone.",
)


### What we just learned

- Colour tells us *which* community a character belongs to — but the communities aren't spatially separated.
- A reader scanning the figure cannot see any of the Book 1 arcs. Purple dots (Daenerys) are next to blue dots (Riverlands); orange dots (Sansa) sit mixed with gold (King's Landing).
- The shared layout is the *right* choice for labels (notebook 03 proved that), but it's the *wrong* choice for showing community structure.

**The next section shows an alternative layout built specifically to make communities legible.**


## Section B — A Community-Aware Ring Layout

If the shared layout can't show communities, we need a different layout for that job. The standard industry solution is a **two-stage hierarchical layout** in the Gephi tradition:

1. **Stage 1 — meta-graph on a ring with variable spacing.** Each community becomes a single super-node. We arrange them on a ring in a greedy TSP order (strongly-interacting communities become adjacent) and allocate angular space proportional to community size: small communities get narrow arcs, large ones get wider ones.
2. **Stage 2 — concentric rings inside each disk.** Each community's members sit on concentric rings. Node size encodes weighted degree, and the highest-degree members are placed on the rim *facing the figure centre* — so inter-community edges emerge from the short (interior) side of each disk.
3. **Soft circular backgrounds** instead of convex-hull polygons. Circles are smooth, have no corners, and match the industry look (Gephi, BubbleSets, GMap).

We'll build it in three small steps.


### Step 1 — Compute the layout

`community_aware_layout` returns three things:

- `pos` — a `{node: (x, y)}` dict ready for `nx.draw_networkx_*`
- `super_pos` — the centre of each community's disk, used to draw the background and place legend-style labels
- `radii` — the radius of each community disk in data coordinates, with disk *area* proportional to member count (so a 48-node arc is visibly bigger than a 3-node one)


In [ ]:
community_aware_pos, community_super_pos, community_radii = community_aware_layout(
    GOT, node_to_cid, communities,
    size_proportional=True,
    gap_fraction=0.14,
)

# Spot-check: radii should grow with community size.
pd.DataFrame({
    "community": [community_names[c] for c in community_radii],
    "size": [len(communities[c]) for c in community_radii],
    "radius": [round(r, 3) for r in community_radii.values()],
}).sort_values("size", ascending=False).reset_index(drop=True)


### Step 2 — Prepare per-node encodings (colour, size, edge groups)

With the layout fixed, we now compute the visual channels:

- **colour** — community fill (same as Section A, now meaningful because position agrees with colour)
- **size** — weighted degree, `minmax`-scaled into a readable marker-size range
- **edge groups** — intra-community edges tinted by community colour; inter-community edges drawn in dark slate so brokerage is prominent


In [ ]:
weighted_degree = dict(GOT.degree(weight="weight"))
node_sizes = dict(zip(
    GOT.nodes(),
    minmax_scale([weighted_degree[n] for n in GOT.nodes()], low=30, high=240),
))

edge_widths = minmax_scale(
    [d["weight"] for _, _, d in GOT.edges(data=True)],
    low=0.35, high=3.0,
)

# Split edges into per-community intra groups and a single inter group.
intra_by_cid = {cid: ([], []) for cid in set(node_to_cid.values())}
inter_edges, inter_widths = [], []
for i, (u, v, _) in enumerate(GOT.edges(data=True)):
    cu, cv = node_to_cid[u], node_to_cid[v]
    if cu == cv:
        intra_by_cid[cu][0].append((u, v))
        intra_by_cid[cu][1].append(edge_widths[i])
    else:
        inter_edges.append((u, v))
        inter_widths.append(edge_widths[i])

# Pack the ring-drawing arguments once — B, C, and D all pass this same
# bundle to `draw_community_ring`, so any future style tweak happens in
# one place.
RING_KW = dict(
    pos=community_aware_pos,
    super_pos=community_super_pos,
    radii=community_radii,
    palette=palette,
    node_to_cid=node_to_cid,
    intra_by_cid=intra_by_cid,
    inter_edges=inter_edges,
    inter_widths=inter_widths,
    node_sizes=node_sizes,
)
LABEL_KW = dict(
    super_pos=community_super_pos,
    radii=community_radii,
    pos=community_aware_pos,
    anchors=dict(enumerate(community_anchors)),
    palette=palette,
)

print(f"{sum(len(eds) for eds, _ in intra_by_cid.values())} intra-community edges, "
      f"{len(inter_edges)} inter-community edges")


### Step 3 — Draw the figure

The heavy lifting is done by two helper calls:

1. `draw_community_ring(ax, GOT, **RING_KW)` — soft discs + community-tinted intra edges + slate inter-community edges + community-coloured nodes, all in one pass.
2. `add_community_legend(...)` — size-sorted legend in the upper-right.

Sections C and D reuse the exact same `RING_KW` bundle, so the three figures are guaranteed to share every style setting.


In [ ]:
fig, ax = plt.subplots(figsize=(13.0, 11.0))
ax.set_aspect("equal")

draw_community_ring(ax, GOT, **RING_KW)
fit_network_axes(ax, community_aware_pos, pad=0.05)
add_community_legend(ax, communities=communities, community_names=community_names,
                     palette=palette, cids=community_super_pos.keys())

add_panel_header(
    ax,
    "B. Community-aware ring layout (Gephi-style)",
    "Disk area is proportional to the number of members; small communities occupy narrower angular slots than large ones. Within each disk, members sit on concentric rings with the highest-degree on the rim facing the figure centre, so inter-community edges emerge from the short side of each disk.",
)


### What we just learned

- **Position now agrees with colour.** Each community is a visually separate island; the group structure is readable at a glance.
- **Size reveals centrality within groups.** The biggest dot on each disk's interior rim is its anchor character (Eddard, Tyrion, Jon, etc.) — exactly the node the inter-community edges originate from.
- **Variable ring spacing honours the data.** A 3-character sub-clique (House Frey, the Prologue rangers) tucks into a narrow slot; the 48-character Riverlands arc gets the room it deserves.

What's still missing: **textual names.** The legend on the side works but forces the reader to jump between colour and text. Section C fixes that with one anchor label per community.


## Section C — Add One Anchor Label per Community

Section B showed the community structure clearly, but no community carried a textual name on the figure itself — a reader had to cross-reference the legend to know which arc they were looking at.

A publication-ready version needs at least one **textual anchor per community** on the drawing. The cheapest and clearest way: label each community's **anchor character** (the highest-weighted-degree member, already computed in the preliminaries).

The labels all sit on a **single outer ring** beyond every community disc — not on each disc's own border. Using a single ring is what keeps small and large communities on the same label radius, prevents a small community's label from crashing into a neighbouring large disc, and gives leader lines a clean radial fan. A leader line joins each label to its anchor node on the inward-facing rim.


### Step 1 — Draw the ring layout and attach anchor labels

The draw path is Section B's cell with one extra line: `add_outer_ring_labels(ax, pad=1.0, **LABEL_KW)`. The `LABEL_KW` bundle was packed in Section B's step 2, so the call site stays tidy and the same label style carries over to Section D's composite.


In [ ]:
fig, ax = plt.subplots(figsize=(13.0, 11.0))
ax.set_aspect("equal")

# Section B's figure + the one thing it was missing: textual anchors.
draw_community_ring(ax, GOT, **RING_KW)
fit_network_axes(ax, community_aware_pos, pad=0.25)
add_outer_ring_labels(ax, pad=1.0, **LABEL_KW)
add_community_legend(ax, communities=communities, community_names=community_names,
                     palette=palette, cids=community_super_pos.keys())

add_panel_header(
    ax,
    "C. Community-aware ring layout with anchor labels",
    "Section B, plus one anchor label per community. All labels sit on a single outer ring beyond every community disc, each on the radial line from the figure centre through its community — so labels always clear every disc regardless of community size.",
)


### What we just added

- Every community now carries a **named anchor** directly on the figure — no more colour-legend hopping.
- Labels sit on the **outer** side of each disc while the inter-community edges emerge from the **inner** rim, so text and edges never fight for the same ink.
- Leader lines are faint and short — they just confirm which node a label refers to, nothing more.

This is now our canonical community figure. The flagship composite in Section D uses it as the main panel.


## Section D — The Flagship Composite

We combine the ring-layout-with-labels (from Section C) with three supporting panels into one publication-ready figure:

| panel | placement | content |
|---|---|---|
| **A** main | left, square | Section C's figure: ring layout + outer-ring anchor labels |
| **B** histogram | top-right | weighted-degree distribution with the median |
| **C** ranking | bottom-right | top-10 characters by weighted degree (horizontal lollipop) |
| **D** named communities | bottom band (full width) | single-row colour legend mapping each community name to its anchor |

The key compositional move: the named-communities legend sits as a **thin horizontal band under the whole figure** (height ratio ≈ 8:1). This is the standard layout used by data journalism (NYT, FT, The Economist) when a categorical legend has more than a handful of entries — a horizontal band reads left-to-right like a caption and doesn't steal vertical space from the main panel.


### Step 1 — Prepare the ranking + reference data

The main panel's visual ingredients (positions, radii, edge groups, node sizes, community names/anchors) are all already computed in Section B. We just need the top-10 ranking for the bottom-right panel.


In [ ]:
interaction_strength = pd.Series(weighted_degree).sort_values(ascending=False)
top10_ranked = interaction_strength.head(10).sort_values()  # ascending for horizontal lollipop
top10_ranked


### Step 2 — Compose the figure

Four panels on one canvas. The main panel is three lines of drawing code — the same `draw_community_ring` + `add_outer_ring_labels` calls Sections B and C used — so there is zero style drift between Section C's standalone figure and the A panel of the composite.


In [ ]:
FLAGSHIP_SIZE = (15.5, 12.5)

fig = plt.figure(figsize=FLAGSHIP_SIZE)

# Outer 2-row grid (main/side + wide legend band); top row splits into
# main panel + stacked hist/rank.
outer = fig.add_gridspec(
    2, 1,
    height_ratios=[8, 1],
    left=0.045, right=0.985, top=0.88, bottom=0.06,
    hspace=0.10,
)
top_gs = outer[0].subgridspec(1, 2, width_ratios=[1.7, 1.0], wspace=0.16)
right_gs = top_gs[0, 1].subgridspec(2, 1, hspace=0.55)

ax_main = fig.add_subplot(top_gs[0, 0])
ax_hist = fig.add_subplot(right_gs[0, 0])
ax_rank = fig.add_subplot(right_gs[1, 0])
ax_guide = fig.add_subplot(outer[1, 0])
ax_main.set_aspect("equal")

# ---- A. Main panel: the Section C figure, built from the same helpers.
draw_community_ring(ax_main, GOT, **RING_KW)
fit_network_axes(ax_main, community_aware_pos, pad=0.25)
add_outer_ring_labels(ax_main, pad=1.0, **LABEL_KW)
add_panel_header(
    ax_main,
    "A. Communities as spatial islands",
    "Disk area is proportional to member count; every anchor label sits on a single outer ring beyond the disc envelope.",
)

# ---- B. Histogram of interaction strength.
ax_hist.hist(
    interaction_strength.values,
    bins=20,
    color=lighten_color(DV_PALETTE["blue"], 0.35),
    edgecolor="#1F2933", linewidth=0.6,
)
median_val = float(interaction_strength.median())
ax_hist.axvline(median_val, color=DV_PALETTE["red"], linewidth=1.4, linestyle="--")
ax_hist.text(
    median_val, ax_hist.get_ylim()[1] * 0.88,
    f"  median = {median_val:.0f}",
    color=DV_PALETTE["red"], fontsize=TEXT["annotation"] * 0.9,
    va="top", ha="left",
)
ax_hist.set_title("B. Interaction strength",
                  fontsize=TEXT["panel_title"], loc="left", pad=12)
ax_hist.set_xlabel("Weighted degree", fontsize=TEXT["annotation"], labelpad=4)
ax_hist.set_ylabel("Characters", fontsize=TEXT["annotation"], labelpad=4)
ax_hist.tick_params(labelsize=TEXT["annotation"] * 0.9)
ax_hist.grid(axis="y", alpha=0.12)

# ---- C. Top-10 ranking.
ax_rank.hlines(
    y=top10_ranked.index, xmin=0, xmax=top10_ranked.values,
    color=lighten_color(DV_PALETTE["blue"], 0.75), linewidth=2.2,
)
ax_rank.scatter(
    top10_ranked.values, top10_ranked.index,
    s=60, color=DV_PALETTE["blue"], edgecolor="#1F2933", zorder=3,
)
for name, value in top10_ranked.items():
    ax_rank.annotate(
        f" {value:.0f}", xy=(value, name), xytext=(4, 0),
        textcoords="offset points", ha="left", va="center",
        fontsize=TEXT["annotation"] * 0.85, color="#1F2933",
    )
ax_rank.set_title("C. Top-10 by weighted degree",
                  fontsize=TEXT["panel_title"], loc="left", pad=12)
ax_rank.set_xlabel("Weighted degree", fontsize=TEXT["annotation"], labelpad=4)
ax_rank.tick_params(axis="y", labelsize=TEXT["annotation"] * 0.85)
ax_rank.tick_params(axis="x", labelsize=TEXT["annotation"] * 0.85)
ax_rank.grid(axis="x", alpha=0.12)
ax_rank.set_xlim(0, top10_ranked.max() * 1.22)

# ---- D. Named-communities band (full width, below everything).
# Horizontal legend: one column per community, swatch on the left, name
# over anchor on the right. Sorted descending by size so the reader scans
# from the most central arcs down to the smallest.
ax_guide.axis("off")
ax_guide.set_xlim(0, 1)
ax_guide.set_ylim(0, 1)
ax_guide.text(
    0.0, 1.15, "D. Named communities",
    transform=ax_guide.transAxes,
    ha="left", va="top",
    fontsize=TEXT["panel_title"], color="#111111",
)

n_comms = len(communities)
guide_order = sorted(range(n_comms), key=lambda c: -len(communities[c]))
slot_width = 1.0 / n_comms
for slot, cid in enumerate(guide_order):
    x0 = slot * slot_width
    swatch_x = x0 + slot_width * 0.04
    text_x = x0 + slot_width * 0.14
    ax_guide.scatter(
        swatch_x, 0.62, s=160,
        color=palette[cid], edgecolor="black", linewidth=0.7,
        transform=ax_guide.transAxes, clip_on=False,
    )
    ax_guide.text(
        text_x, 0.78, community_names[cid],
        transform=ax_guide.transAxes,
        ha="left", va="center",
        fontsize=TEXT["annotation"] * 0.95,
        fontweight="semibold", color="#111111",
    )
    ax_guide.text(
        text_x, 0.42,
        f"{community_anchors[cid]} · {len(communities[cid])} chars",
        transform=ax_guide.transAxes,
        ha="left", va="center",
        fontsize=TEXT["annotation"] * 0.80,
        color=DV_PALETTE["gray"],
    )

# ---- Master title + subtitle + credit.
fig.text(
    0.045, 0.955,
    "A Song of Ice and Data - Book 1 character network",
    ha="left", va="top",
    fontsize=TEXT["figure_title"], fontweight="semibold", color="#0F1A24",
)
fig.text(
    0.045, 0.920,
    "Weighted Louvain recovers Book 1's main arcs; each community is drawn\n"
    "as a sized disc with its anchor character labelled.",
    ha="left", va="top",
    fontsize=TEXT["annotation"] * 0.95, color=DV_PALETTE["gray"],
)
fig.text(
    0.985, 0.015,
    "Data: Beveridge & Shan (2016). Layout: two-stage community-aware ring.   "
    "Complex Network Dataviz, 2025-26.",
    ha="right", va="bottom",
    fontsize=TEXT["annotation"] * 0.82, color=DV_PALETTE["gray"], style="italic",
)


### Why this composite works

- **One geometry across all panels**: the main panel shows the community-aware ring layout, and the three side panels read distributions / names / rankings off the *same* underlying partition. Every colour, every community name, every member count in the side panels corresponds exactly to something visible in the main panel.
- **Anchor labels replace a separate brokerage step**: because the highest-degree member of each community is labelled on the main panel *and* repeated in the named-communities guide, a reader can triangulate between them without scanning the graph for specific characters.
- **Supporting panels absorb the clutter that a labelled dense graph would generate**: histogram for the "how skewed is interaction strength" question, guide for "which colour means what", ranking for "who's most central overall".


## Final Takeaway

Five questions a student should be able to answer after this notebook:

1. Why does a layout optimised for *labels* (notebook 03's overlap-free Graphviz) fail to show community structure?
2. What does a community-aware layout encode that colour alone cannot?
3. Why do we place anchors on the inward-facing rim of each community disc?
4. How does variable angular spacing on the ring preserve the size-of-community signal?
5. What is the minimum labelling we need to go from an abstract community figure to a narrative one?

The rule this notebook comes back to:

**One question, one layout. One label per group, placed where it doesn't fight the graph.**
